# Round 3: new backbones + saved probabilities (decision-support analysis)

**What this does**: the same 20 folds as the previous two rounds (reusing `runs_phase0/folds.csv` and the resized image cache), the same two-stage training, the same two seeds.
Only the backbone changes, and this time **the probability of every class for every image is saved** (for both the validation and test sets), to support the "defer to the grower when unsure" analysis.

| Config | Backbone | Year | Parameters | Source |
|---|---|---|---|---|
| `effb0` | EfficientNet-B0 (reference, rerun) | 2019 | 4.0 M | torchvision |
| `mnv3l` | MobileNetV3-Large | 2019 | 4.2 M | timm |
| `mnv4m` | MobileNetV4-Conv-Medium | 2024 | 8.4 M | timm |
| `cnxv2t` | ConvNeXt V2-Tiny | 2023 | 27.9 M | timm |
| `vits` | ViT-Small/16 | 2021 | 21.7 M | timm |

5 configs x 20 folds x 2 seeds = 200 runs. Run the three lightweight ones first (about 3-4 hours), then ConvNeXt V2 and ViT (about 4-5 hours), for **about 8 hours in total** (A100).

**Differences from the previous two rounds** (all written into `PREREG.md` in cell 4)
- Each backbone uses its own ImageNet normalisation (0.5/0.5/0.5 for ViT, the standard ImageNet values for the others); the input is always 224 px (MobileNetV4 was pretrained at 256 px, but 224 is used here too)
- The classification head is always Dropout 0.3 + Linear (the same as the original EfficientNet-B0 head)
- The training DataLoader uses `drop_last=True`: MobileNetV4 has a BatchNorm before its head, which raises an error if the last batch has only 1 image.
  To keep the comparison fair, the reference `effb0` is also rerun under the same setting
- `preds.csv` gains a `role` column (val / test) and 4 probability columns `p_<class>`

**Decision rules fixed in advance** (cell 4 writes them to `PREREG.md` on Drive together with the time, before any run):
The primary comparisons are each of the 4 new backbones against `effb0`, paired (primary scoring: video frames excluded), using the corrected resampled t-test with Holm correction.
It counts as an improvement only if the mean improvement is >= 2 points and the Holm-adjusted p < 0.05. The "defer to the grower" threshold is set on the inner validation set only; the test set is used only for reporting. Results are reported as they are, whatever they show.

Interruptions are fine: each run is written to Drive as soon as it finishes, and "Run all" again skips the finished ones.

Before running: choose **GPU** under "Runtime -> Change runtime type" (A100 or L4 both work).
When done: run the last two cells (compute cost, analysis), send the whole output to Claude, and send along `results.csv`, `preds.csv`, `cost.csv`, `PREREG.md` and `versions.txt` from the `runs_v3` folder.

## 1. Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or '⚠ No GPU! Change the runtime type')

## 2. Install timm (pinned version, for reproducibility)

In [ ]:
!pip -q install timm==1.0.9

## 3. Settings

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/p1_reconstructure')
SRC_DIR   = DRIVE_DIR / 'runs_phase0'      # round 1's cache_512.pkl, folds.csv and results.csv are all here
OUT_DIR   = DRIVE_DIR / 'runs_v3'          # results of this round are written here

# config name: (source, model name)
BACKBONES = {
    'effb0':  ('torchvision', 'efficientnet_b0'),
    'mnv3l':  ('timm', 'mobilenetv3_large_100.ra_in1k'),
    'mnv4m':  ('timm', 'mobilenetv4_conv_medium.e500_r256_in1k'),
    'cnxv2t': ('timm', 'convnextv2_tiny.fcmae_ft_in22k_in1k'),
    'vits':   ('timm', 'vit_small_patch16_224.augreg_in21k_ft_in1k'),
}
REFERENCE = 'effb0'
PRIMARY = ['mnv3l', 'mnv4m', 'cnxv2t', 'vits']          # all compared with effb0
PHASES = [['effb0', 'mnv3l', 'mnv4m'], ['cnxv2t', 'vits']]
SIZE = 224
SEEDS = [0, 1]
EPOCHS_S1, EPOCHS_S2 = 15, 15
LR_S1, LR_S2 = 1e-3, 1e-5
BATCH = 16
WORKERS = 2
TARGETS = [0.90, 0.95]   # "defer to the grower" analysis: target accuracy for the part the model decides itself
PRETRAINED = True
QUICK_TEST = False       # True = 1 epoch per stage and only 2 folds, a few-minute trial to check that it runs

## 4. Write down the pre-specified rules and software versions (written only on the first run, never overwritten)

In [ ]:
import datetime, platform, sys
import torch, torchvision, timm, sklearn
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREREG = OUT_DIR / 'PREREG.md'
if not PREREG.exists() and not QUICK_TEST:
    PREREG.write_text(f'''# Pre-specified analysis, written {datetime.datetime.now().isoformat(timespec='seconds')} before any v3 run

Folds: runs_phase0/folds.csv (20 grouped folds, as in the first experiment). Seeds {SEEDS}.
Training protocol identical to the first experiment (two stages, {EPOCHS_S1}+{EPOCHS_S2} epochs, Adam {LR_S1} then {LR_S2},
batch {BATCH}, class-weighted loss and sampler, same augmentation, checkpoint by inner-validation macro F1), except:
- backbone (below); classification head Dropout(0.3) + Linear for every backbone; input {SIZE} px for every backbone
  (MobileNetV4-Conv-Medium was pretrained at 256 px but is fine-tuned and tested at {SIZE} px like the others);
- per-backbone normalisation from timm's pretrained config (torchvision ImageNet values for effb0);
- training DataLoader with drop_last=True (MobileNetV4 has a BatchNorm in its head); effb0 is rerun under the same setting.

Configurations: {BACKBONES}

Primary metric: macro F1 over classes present in each test fold, seeds averaged within fold.
Primary scoring: the 63 Phomopsis video frames excluded; all test images reported as secondary.
Primary comparisons (each vs {REFERENCE}): {PRIMARY}.
Test: Nadeau-Bengio corrected resampled t-test over 20 folds, factor 1/20 + mean n_test / mean (n_train + n_val).
Multiplicity: Holm over the {len(PRIMARY)} primary comparisons.
Counted as an improvement only if mean paired difference >= 2 pp AND Holm-adjusted p < 0.05.

Decision-support (selective prediction) analysis, per run:
- confidence = maximum softmax probability;
- the threshold is chosen on the inner-validation set only: the smallest confidence threshold at which the accepted
  validation images reach the target accuracy ({TARGETS}); if no threshold reaches it, every image is deferred;
- on the test fold we report coverage (share decided by the model), accuracy on the accepted images, and the share of each
  class deferred to the grower; also the area under the risk-coverage curve (AURC) and top-2 accuracy;
- summaries are means and SDs over the 40 runs of each backbone; no significance test is planned for these.
Secondary (descriptive): effb0 here vs 'none' in runs_phase0 (rerun agreement); parameters, MACs-free CPU latency.
All results are reported whatever their direction.
''', encoding='utf-8')
print(PREREG.read_text(encoding='utf-8') if PREREG.exists() else '(quick-test mode, PREREG not written)')
(OUT_DIR / 'versions.txt').write_text(
    f'python {platform.python_version()}\ntorch {torch.__version__}\ntorchvision {torchvision.__version__}\n'
    f'timm {timm.__version__}\nscikit-learn {sklearn.__version__}\n'
    f'gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}\n', encoding='utf-8')
print((OUT_DIR / 'versions.txt').read_text())

## 5. Load the image cache and the 20 folds from round 1

In [ ]:
import io, re, pickle, time, copy, random
import numpy as np
import pandas as pd
from PIL import Image

CLASSES_ALL = ['Algal', 'Leaf_rot', 'Phomopsis', 'Pink_disease', 'Root_disease']


def norm_class(part):
    if part.startswith('Leaf_rot'):
        return 'Leaf_rot'
    for c in CLASSES_ALL:
        if part.lower() == c.lower():
            return c
    return None


with open(SRC_DIR / 'cache_512.pkl', 'rb') as f:
    cache = pickle.load(f)
assert len(cache['images']) == 560, len(cache['images'])
meta = pd.read_csv(io.StringIO(cache['sessions_csv']))
meta['cls'] = meta['cls'].map(norm_class)
meta = meta[meta['cls'] != 'Pink_disease'].reset_index(drop=True)
CLASSES = sorted(meta['cls'].unique(), key=CLASSES_ALL.index)
meta['y'] = meta['cls'].map({c: i for i, c in enumerate(CLASSES)})
meta['video'] = meta['session'].astype(str).str.startswith('video:')
folds = pd.read_csv(SRC_DIR / 'folds.csv')
assert len(meta) == 550 and folds.groupby(['repeat', 'fold']).size().eq(550).all()
print(f'{len(meta)} images ({meta.video.sum()} of them video frames), classes {CLASSES}, {folds.groupby(["repeat", "fold"]).ngroups} folds')

## 6. Model (backbone + Dropout 0.3 + Linear)

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class Net(nn.Module):
    def __init__(self, body, n_feat, n_cls):
        super().__init__()
        self.body = body
        self.head = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(n_feat, n_cls))

    def forward(self, x):
        return self.head(self.body(x))


def build_model(cfg, n_cls, pretrained=True):
    # Returns (model, normalisation mean, std). Stage 1 trains only the head; the backbone is frozen.
    src, name = BACKBONES[cfg]
    if src == 'torchvision':
        m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        body = nn.Sequential(m.features, m.avgpool, nn.Flatten(1))     # identical to the original EfficientNet-B0
        n_feat, mean, std = m.classifier[1].in_features, IMAGENET_MEAN, IMAGENET_STD
    else:
        body = timm.create_model(name, pretrained=pretrained, num_classes=0)   # num_classes=0 -> outputs the pooled features
        body.eval()
        with torch.no_grad():                                   # one forward pass to get the feature dimension (MobileNet's num_features is not the final output dimension)
            n_feat = body(torch.zeros(1, 3, SIZE, SIZE)).shape[1]
        body.train()
        dc = timm.data.resolve_model_data_config(body)
        mean, std = tuple(dc['mean']), tuple(dc['std'])
    net = Net(body, n_feat, n_cls)
    for p in net.body.parameters():
        p.requires_grad = False
    return net, mean, std


for cfg in BACKBONES:
    mm, mean, std = build_model(cfg, 4, pretrained=False)
    mm.eval()
    with torch.no_grad():
        out = mm(torch.zeros(2, 3, SIZE, SIZE))
    n_par = sum(p.numel() for p in mm.parameters())
    print(f'{cfg:<7} params {n_par / 1e6:5.2f} M  output {tuple(out.shape)}  mean {mean}')

## 7. Training function (same as the previous two rounds, except the backbone and normalisation can vary)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


def make_tfs(mean, std, size=SIZE):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 1.5)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.10)),
    ])
    test_tf = transforms.Compose([transforms.Resize(round(size * 256 / 224)), transforms.CenterCrop(size),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_tf, test_tf


class ImageSet(Dataset):
    def __init__(self, ims, ys, tf):
        self.ims, self.ys, self.tf = list(ims), list(ys), tf

    def __len__(self):
        return len(self.ims)

    def __getitem__(self, i):
        return self.tf(self.ims[i]), int(self.ys[i])


def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _worker_init(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out, ys = [], []
    for X, y in loader:
        with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            out.append(model(X.to(DEVICE)).float().cpu())
        ys += y.tolist()
    return torch.cat(out).numpy(), np.array(ys)


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def macro_f1_present(yt, yp):
    return f1_score(yt, yp, labels=sorted(set(yt)), average='macro', zero_division=0)


def train_model(cfg, ims_tr, y_tr, ims_va, y_va, n_cls, seed, epochs, pretrained=True):
    seed_everything(seed)
    model, mean, std = build_model(cfg, n_cls, pretrained)
    model = model.to(DEVICE)
    train_tf, test_tf = make_tfs(mean, std)
    y_tr = np.asarray(y_tr)
    counts = np.bincount(y_tr, minlength=n_cls)
    cw = torch.tensor(len(y_tr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
    sw = (1.0 / np.maximum(counts, 1))[y_tr]
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True, generator=g)
    kw = dict(num_workers=WORKERS, worker_init_fn=_worker_init, pin_memory=(DEVICE == 'cuda'))
    tl = DataLoader(ImageSet(ims_tr, y_tr, train_tf), batch_size=BATCH, sampler=sampler, generator=g,
                    drop_last=True, **kw)
    vl = DataLoader(ImageSet(ims_va, y_va, test_tf), batch_size=64, shuffle=False, **kw)
    crit = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    best = [-1.0, None]
    for n_ep, lr, unfreeze in [(epochs[0], LR_S1, False), (epochs[1], LR_S2, True)]:
        if unfreeze:
            model.load_state_dict(best[1])
            for p in model.parameters():
                p.requires_grad = True
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        for _ in range(n_ep):
            model.train()
            for X, y in tl:
                X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                    loss = crit(model(X), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            logits, yv = predict_logits(model, vl)
            f1v = macro_f1_present(yv, logits.argmax(1))
            if f1v > best[0]:
                best = [f1v, copy.deepcopy(model.state_dict())]
    model.load_state_dict(best[1])
    return model, best[0], test_tf


def to_pil(idx):
    return [Image.open(io.BytesIO(cache['images'][(meta.at[i, 'cls'], meta.at[i, 'file'])])).convert('RGB')
            for i in idx]


def pred_frame(cfg, rep, k, s, role, idx, yt, logits):
    pr = softmax(logits)
    d = pd.DataFrame({'config': cfg, 'repeat': rep, 'fold': k, 'seed': s, 'role': role,
                      'file': meta.file.values[idx], 'cls': meta.cls.values[idx],
                      'y_true': yt, 'y_pred': pr.argmax(1)})
    for j, c in enumerate(CLASSES):
        d[f'p_{c}'] = pr[:, j].round(6)
    return d

## 8. Main loop (interruptible and resumable)

In [ ]:
res_csv, pred_csv = OUT_DIR / ('results_quick.csv' if QUICK_TEST else 'results.csv'), \
                    OUT_DIR / ('preds_quick.csv' if QUICK_TEST else 'preds.csv')
done = set()
if res_csv.exists():
    r0 = pd.read_csv(res_csv)
    done = set(zip(r0.config, r0.repeat, r0.fold, r0.seed))
keys = sorted(folds.groupby(['repeat', 'fold']).groups)[: 2 if QUICK_TEST else None]
jobs = [(c, rep, k, s) for ph in PHASES for (rep, k) in keys for s in SEEDS for c in ph]
todo = [j for j in jobs if j not in done]
epochs = (1, 1) if QUICK_TEST else (EPOCHS_S1, EPOCHS_S2)
print(f'{len(jobs)} runs in total, {len(jobs) - len(todo)} done, {len(todo)} left')
t_start = time.time()
for n, (cfg, rep, k, s) in enumerate(todo, 1):
    f = folds[(folds.repeat == rep) & (folds.fold == k)]
    tr, va, te = (f[f.role == r].idx.values for r in ['train', 'val', 'test'])
    t0 = time.time()
    model, vf1, test_tf = train_model(cfg, to_pil(tr), meta.y.values[tr], to_pil(va), meta.y.values[va],
                                      len(CLASSES), s, epochs, PRETRAINED)
    out = {}
    for role, idx in [('val', va), ('test', te)]:
        dl = DataLoader(ImageSet(to_pil(idx), meta.y.values[idx], test_tf), batch_size=64, shuffle=False,
                        num_workers=WORKERS)
        out[role] = predict_logits(model, dl)
    logits, yt = out['test']
    yp = logits.argmax(1)
    # write preds first, then results: an interruption in between only leaves duplicate rows in preds, which the analysis removes
    pd.concat([pred_frame(cfg, rep, k, s, role, idx, out[role][1], out[role][0])
               for role, idx in [('val', va), ('test', te)]]).to_csv(
        pred_csv, mode='a', header=not pred_csv.exists(), index=False)
    row = {'config': cfg, 'repeat': rep, 'fold': k, 'seed': s, 'n_train': len(tr), 'n_val': len(va),
           'n_test': len(te), 'n_params': sum(p.numel() for p in model.parameters()),
           'val_macro_f1': round(vf1 * 100, 2),
           'test_macro_f1_present': round(macro_f1_present(yt, yp) * 100, 2),
           'test_acc': round(accuracy_score(yt, yp) * 100, 2), 'minutes': round((time.time() - t0) / 60, 2)}
    pd.DataFrame([row]).to_csv(res_csv, mode='a', header=not res_csv.exists(), index=False)
    left = (time.time() - t_start) / n * (len(todo) - n) / 3600
    print(f"[{n}/{len(todo)}] {cfg:<7} rep{rep} fold{k} seed{s}  F1 {row['test_macro_f1_present']:5.1f}"
          f"  ({row['minutes']:.1f} min, about {left:.1f} h left)", flush=True)
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
print('All done')

## 9. Compute cost: parameter count and single-thread CPU latency (measured in the same Colab runtime, written to cost.csv)

In [ ]:
import os
n_threads = torch.get_num_threads()
torch.set_num_threads(1)
rows = []
for cfg in BACKBONES:
    m, _, _ = build_model(cfg, len(CLASSES), pretrained=False)
    m.eval()
    x = torch.zeros(1, 3, SIZE, SIZE)
    with torch.no_grad():
        for _ in range(10):
            m(x)
        ts = []
        for _ in range(50):
            t0 = time.perf_counter()
            m(x)
            ts.append((time.perf_counter() - t0) * 1000)
    rows.append({'config': cfg, 'params_M': round(sum(p.numel() for p in m.parameters()) / 1e6, 3),
                 'cpu_ms_1thr_median': round(float(np.median(ts)), 2),
                 'cpu_ms_1thr_p90': round(float(np.percentile(ts, 90)), 2)})
cost = pd.DataFrame(rows)
cost.to_csv(OUT_DIR / ('cost_quick.csv' if QUICK_TEST else 'cost.csv'), index=False)
torch.set_num_threads(n_threads)
print(cost.to_string(index=False))

## 10. Analysis (send the whole output to Claude)

In [ ]:
from scipy import stats

r = pd.read_csv(res_csv)
p = pd.read_csv(pred_csv).drop_duplicates(['config', 'repeat', 'fold', 'seed', 'role', 'file', 'cls'], keep='last')
p = p.merge(meta[['cls', 'file', 'video']], on=['cls', 'file'], how='left')
PCOLS = [f'p_{c}' for c in CLASSES]
sz = r.groupby(['repeat', 'fold'])[['n_train', 'n_val', 'n_test']].first()
factor = 1 / len(sz) + sz.n_test.mean() / (sz.n_train + sz.n_val).mean()
cfgs = [c for c in BACKBONES if c in set(r.config)]
print('Runs completed per config:', r.config.value_counts().to_dict(), f'  correction factor {factor:.3f}')


def run_metrics(q):
    rows = []
    for (c, rep, k, s), g in q.groupby(['config', 'repeat', 'fold', 'seed']):
        rows.append({'config': c, 'repeat': rep, 'fold': k, 'seed': s,
                     'macro_f1': macro_f1_present(g.y_true, g.y_pred) * 100,
                     'acc': accuracy_score(g.y_true, g.y_pred) * 100})
    return pd.DataFrame(rows)


def corrected(d):
    d = np.asarray(d, float)
    J = len(d)
    se = np.sqrt(factor * d.var(ddof=1))
    h = stats.t.ppf(0.975, J - 1) * se
    return d.mean(), d.mean() - h, d.mean() + h, 2 * stats.t.sf(abs(d.mean() / se), J - 1), int((d > 0).sum()), J


test = p[p.role == 'test']
for label, q in [('video frames excluded (primary)', test[~test.video]), ('all test images', test)]:
    rm = run_metrics(q)
    print(f'\n===== {label} =====')
    print('macro F1 (%, mean ± SD over 40 runs):')
    for c in cfgs:
        v = rm[rm.config == c].macro_f1
        print(f'  {c:<7} {v.mean():6.2f} ± {v.std(ddof=1):5.2f}   acc {rm[rm.config == c].acc.mean():6.2f}   (n={len(v)})')
    pf = rm.groupby(['config', 'repeat', 'fold']).macro_f1.mean().unstack('config')
    rows = []
    for c in PRIMARY:
        if c in pf and REFERENCE in pf:
            d = (pf[c] - pf[REFERENCE]).dropna()
            if len(d) >= 3:
                rows.append((c, *corrected(d)))
    order = sorted(range(len(rows)), key=lambda i: rows[i][4])
    adj, running = [None] * len(rows), 0
    for rank, i in enumerate(order):
        running = max(running, min(1, (len(rows) - rank) * rows[i][4]))
        adj[i] = running
    if rows:
        print(f'Paired comparison with {REFERENCE} (20 folds):')
        for (c, mu, lo, hi, pv, w, J), pa in zip(rows, adj):
            ok = (mu >= 2) and (pa < 0.05) and J == 20
            print(f'  {c:<7} {mu:+7.2f} [{lo:+6.2f}, {hi:+6.2f}]  p {pv:6.3f}  Holm {pa:6.3f}  higher {w}/{J}  '
                  f'{"✅ improved" if ok else "—"}')
    sg = rm.pivot_table(index=['config', 'repeat', 'fold'], columns='seed', values='macro_f1')
    if sg.shape[1] == 2:
        print('Seed-only difference, mean |seed0 − seed1|:',
              (sg.iloc[:, 0] - sg.iloc[:, 1]).abs().groupby(level='config').mean().round(2).to_dict())


def pick_threshold(conf, correct, target):
    # On the validation set, find the smallest confidence threshold at which the accepted images reach accuracy >= target; if none does, defer everything to a human
    for t in np.sort(np.unique(conf)):
        acc = correct[conf >= t]
        if len(acc) and acc.mean() >= target:
            return t
    return np.inf


def selective(q_val, q_test, target):
    out = []
    for key, gt in q_test.groupby(['config', 'repeat', 'fold', 'seed']):
        gv = q_val[(q_val.config == key[0]) & (q_val.repeat == key[1]) & (q_val.fold == key[2]) & (q_val.seed == key[3])]
        t = pick_threshold(gv[PCOLS].max(1).values, (gv.y_true == gv.y_pred).values, target)
        conf = gt[PCOLS].max(1).values
        corr = (gt.y_true == gt.y_pred).values
        acc_mask = conf >= t
        o = {'config': key[0], 'coverage': acc_mask.mean() * 100,
             'sel_acc': corr[acc_mask].mean() * 100 if acc_mask.any() else np.nan}
        for j, c in enumerate(CLASSES):
            m = gt.y_true.values == j
            if m.any():
                o[f'defer_{c}'] = (~acc_mask[m]).mean() * 100
        out.append(o)
    return pd.DataFrame(out)


def aurc_top2(q_test):
    out = []
    for key, gt in q_test.groupby(['config', 'repeat', 'fold', 'seed']):
        pr = gt[PCOLS].values
        conf, corr = pr.max(1), (gt.y_true == gt.y_pred).values
        o = np.argsort(-conf)
        risk = np.cumsum(~corr[o]) / np.arange(1, len(o) + 1)
        top2 = np.argsort(-pr, 1)[:, :2]
        out.append({'config': key[0], 'aurc': risk.mean() * 100,
                    'top2_acc': (top2 == gt.y_true.values[:, None]).any(1).mean() * 100})
    return pd.DataFrame(out)


val = p[p.role == 'val']
for label, qv, qt in [('video frames excluded (primary)', val[~val.video], test[~test.video]), ('all test images', val, test)]:
    print(f'\n===== Decision-support analysis: {label} =====')
    at = aurc_top2(qt).groupby('config').agg(['mean', 'std']).round(2)
    print('AURC (lower is better) and top-2 accuracy (%):')
    print(at.reindex(cfgs).to_string())
    for target in TARGETS:
        sel = selective(qv, qt, target)
        g = sel.groupby('config')
        print(f'\nTarget accuracy {target:.0%} (threshold set on the validation set only):')
        print(f'  {"config":<7} {"model-decided %":>14} {"decided acc. %":>16}   share of each class deferred to the grower %')
        for c in cfgs:
            if c not in g.groups:
                continue
            x = g.get_group(c)
            dfr = '  '.join(f'{cl[:6]} {x[f"defer_{cl}"].mean():5.1f}' for cl in CLASSES if f'defer_{cl}' in x)
            print(f'  {c:<7} {x.coverage.mean():8.1f} ± {x.coverage.std(ddof=1):4.1f}   '
                  f'{x.sel_acc.mean():8.1f} ± {x.sel_acc.std(ddof=1):4.1f}   {dfr}')

old = SRC_DIR / 'results.csv'
if old.exists() and REFERENCE in set(r.config):
    o = pd.read_csv(old)
    o = o[o.config == 'none'].set_index(['repeat', 'fold', 'seed'])
    n2 = r[r.config == REFERENCE].set_index(['repeat', 'fold', 'seed'])
    j = n2.join(o[['test_acc']], rsuffix='_old', how='inner')
    if len(j):
        print(f'\nRerun agreement: mean |Δ| in accuracy between {REFERENCE} and round-1 none is {np.abs(j.test_acc - j.test_acc_old).mean():.2f} points'
              f' ({len(j)} pairs; drop_last differs, so small differences are expected)')
print('\nMean time per run (minutes):', r.groupby('config').minutes.mean().round(2).to_dict())